In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, norm


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

SO_DIR = Path("Stack Overflow")
GH_DIR = Path("GitHub")

OUTPUT_DIR = Path("RQ2/cross_platform_analysis_final")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_PATTERN = "labeled_T*.csv"
QUESTION_TYPES = ["how", "why", "what", "other"]


In [3]:
# ============================================================
# LOAD AND PREPARE DATA
# ============================================================

def normalize_question_type(value):
    """Normalize variations of the four question-type labels."""
    if pd.isna(value):
        return np.nan

    value = (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    mapping = {
        "how": "how",
        "how type": "how",
        "why": "why",
        "why type": "why",
        "what": "what",
        "what type": "what",
        "other": "other",
        "others": "other",
        "other type": "other",
    }

    return mapping.get(value, np.nan)


def load_platform(folder, platform):
    """Load and combine all labelled topic files for one platform."""
    files = sorted(folder.rglob(FILE_PATTERN))

    if not files:
        raise FileNotFoundError(
            f"No files matching {FILE_PATTERN!r} were found in "
            f"{folder.resolve()}"
        )

    frames = []

    for file in files:
        df = pd.read_csv(file)

        if "question_type" not in df.columns:
            raise ValueError(
                f"{file} does not contain the required "
                "'question_type' column."
            )

        output = pd.DataFrame(
            {
                "platform": platform,
                "question_type": df["question_type"].map(
                    normalize_question_type
                ),
                "topic": (
                    df["Topic"]
                    if "Topic" in df.columns
                    else np.nan
                ),
                "source_file": file.name,
            }
        )

        frames.append(output)

    data = pd.concat(frames, ignore_index=True)

    invalid_count = data["question_type"].isna().sum()

    if invalid_count:
        print(
            f"{platform}: excluded {invalid_count:,} rows with "
            "missing or unrecognized question-type labels."
        )

    data = data.dropna(subset=["question_type"]).copy()

    print(
        f"{platform}: loaded {len(data):,} artifacts from "
        f"{len(files)} files."
    )

    return data


In [4]:
# ============================================================
# STATISTICAL FUNCTIONS
# ============================================================

def adjusted_standardized_residuals(observed, expected):
    """Calculate adjusted standardized residuals."""
    observed = np.asarray(observed, dtype=float)
    expected = np.asarray(expected, dtype=float)

    total = observed.sum()
    row_proportions = observed.sum(axis=1, keepdims=True) / total
    column_proportions = observed.sum(axis=0, keepdims=True) / total

    denominator = np.sqrt(
        expected
        * (1 - row_proportions)
        * (1 - column_proportions)
    )

    return (observed - expected) / denominator


def calculate_odds_ratio(a, b, c, d):
    """
    Calculate Stack Overflow-versus-GitHub odds ratio and 95% CI.

    OR > 1 indicates a stronger association with Stack Overflow.
    OR < 1 indicates a stronger association with GitHub.
    """
    cells = np.array([a, b, c, d], dtype=float)

    # Apply a continuity correction only if a cell is zero.
    if np.any(cells == 0):
        cells += 0.5

    a, b, c, d = cells

    odds_ratio = (a * d) / (b * c)
    standard_error = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    log_odds_ratio = np.log(odds_ratio)

    lower_ci = np.exp(log_odds_ratio - 1.96 * standard_error)
    upper_ci = np.exp(log_odds_ratio + 1.96 * standard_error)

    z_value = log_odds_ratio / standard_error
    p_value = 2 * norm.sf(abs(z_value))

    return odds_ratio, lower_ci, upper_ci, p_value


def holm_adjustment(p_values):
    """Apply Holm correction to multiple p-values."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ordered = p_values[order]

    adjusted_ordered = np.empty_like(ordered)
    running_maximum = 0.0
    number_of_tests = len(p_values)

    for index, p_value in enumerate(ordered):
        adjusted_value = (number_of_tests - index) * p_value
        running_maximum = max(running_maximum, adjusted_value)
        adjusted_ordered[index] = min(running_maximum, 1.0)

    adjusted = np.empty_like(adjusted_ordered)
    adjusted[order] = adjusted_ordered

    return adjusted


In [5]:
# ============================================================
# LOAD ALL STACK OVERFLOW AND GITHUB FILES
# ============================================================

so_data = load_platform(SO_DIR, "Stack Overflow")
github_data = load_platform(GH_DIR, "GitHub")

combined_data = pd.concat(
    [so_data, github_data],
    ignore_index=True,
)

combined_data.to_csv(
    OUTPUT_DIR / "combined_artifacts.csv",
    index=False,
)

print(f"Combined sample size: {len(combined_data):,}")


Stack Overflow: loaded 495 artifacts from 9 files.
GitHub: loaded 9,116 artifacts from 13 files.
Combined sample size: 9,611


In [6]:
# ============================================================
# CONTINGENCY TABLE
# ============================================================

counts = pd.crosstab(
    combined_data["platform"],
    combined_data["question_type"],
)

counts = counts.reindex(
    index=["Stack Overflow", "GitHub"],
    columns=QUESTION_TYPES,
    fill_value=0,
)

percentages = counts.div(counts.sum(axis=1), axis=0) * 100

print("\nObserved counts")
print(counts)

print("\nPlatform percentages")
print(percentages.round(2))

counts.to_csv(
    OUTPUT_DIR / "question_type_counts.csv"
)

percentages.to_csv(
    OUTPUT_DIR / "question_type_percentages.csv"
)


# ============================================================
# CHI-SQUARE TEST AND CRAMÉR'S V
# ============================================================

chi_square, chi_square_p, degrees_of_freedom, expected_counts = (
    chi2_contingency(
        counts,
        correction=False,
    )
)

sample_size = int(counts.to_numpy().sum())

cramers_v = np.sqrt(
    chi_square
    / (
        sample_size
        * min(
            counts.shape[0] - 1,
            counts.shape[1] - 1,
        )
    )
)

print("\nOverall cross-platform test")
print(f"Chi-square: {chi_square:.4f}")
print(f"Degrees of freedom: {degrees_of_freedom}")
print(f"p-value: {chi_square_p:.8g}")
print(f"Cramér's V: {cramers_v:.4f}")


# ============================================================
# ADJUSTED STANDARDIZED RESIDUALS
# ============================================================

residuals = pd.DataFrame(
    adjusted_standardized_residuals(
        counts.to_numpy(),
        expected_counts,
    ),
    index=counts.index,
    columns=counts.columns,
)

print("\nAdjusted standardized residuals")
print(residuals.round(3))

residuals.to_csv(
    OUTPUT_DIR / "adjusted_standardized_residuals.csv"
)


# ============================================================
# ODDS RATIOS AND 95% CONFIDENCE INTERVALS
# ============================================================

so_total = counts.loc["Stack Overflow"].sum()
github_total = counts.loc["GitHub"].sum()

effect_size_rows = []

for question_type in QUESTION_TYPES:
    so_selected = counts.loc["Stack Overflow", question_type]
    github_selected = counts.loc["GitHub", question_type]

    so_other = so_total - so_selected
    github_other = github_total - github_selected

    odds_ratio, lower_ci, upper_ci, p_value = calculate_odds_ratio(
        so_selected,
        so_other,
        github_selected,
        github_other,
    )

    effect_size_rows.append(
        {
            "question_type": question_type.title(),
            "so_n": int(so_selected),
            "so_percent": 100 * so_selected / so_total,
            "github_n": int(github_selected),
            "github_percent": 100 * github_selected / github_total,
            "difference_percentage_points": (
                100
                * (
                    so_selected / so_total
                    - github_selected / github_total
                )
            ),
            "odds_ratio_so_vs_github": odds_ratio,
            "ci_95_lower": lower_ci,
            "ci_95_upper": upper_ci,
            "p_value": p_value,
        }
    )

effect_sizes = pd.DataFrame(effect_size_rows)

effect_sizes["holm_adjusted_p"] = holm_adjustment(
    effect_sizes["p_value"]
)

effect_sizes["significant_after_holm"] = (
    effect_sizes["holm_adjusted_p"] < 0.05
)

print("\nQuestion-type effect sizes")
print(effect_sizes.round(4).to_string(index=False))

effect_sizes.to_csv(
    OUTPUT_DIR / "question_type_effect_sizes.csv",
    index=False,
)


# ============================================================
# MANUSCRIPT-READY STATISTICAL SUMMARY
# ============================================================

formatted_p = (
    "p < 0.001"
    if chi_square_p < 0.001
    else f"p = {chi_square_p:.3f}"
)

summary = (
    "The distribution of question types differed significantly "
    "between Stack Overflow and GitHub, "
    f"chi-square({degrees_of_freedom}, N = {sample_size:,}) = "
    f"{chi_square:.2f}, {formatted_p}, "
    f"Cramér's V = {cramers_v:.3f}."
)

print("\nManuscript-ready overall result")
print(summary)

with open(
    OUTPUT_DIR / "statistical_summary.txt",
    "w",
    encoding="utf-8",
) as output_file:
    output_file.write(summary + "\n")


# ============================================================
# LATEX TABLE
# ============================================================

latex_data = effect_sizes.copy()

latex_data["Stack Overflow, n (\\%)"] = latex_data.apply(
    lambda row: (
        f"{row['so_n']:,} "
        f"({row['so_percent']:.2f}\\%)"
    ),
    axis=1,
)

latex_data["GitHub, n (\\%)"] = latex_data.apply(
    lambda row: (
        f"{row['github_n']:,} "
        f"({row['github_percent']:.2f}\\%)"
    ),
    axis=1,
)

latex_data["OR (95\\% CI)"] = latex_data.apply(
    lambda row: (
        f"{row['odds_ratio_so_vs_github']:.2f} "
        f"({row['ci_95_lower']:.2f}--"
        f"{row['ci_95_upper']:.2f})"
    ),
    axis=1,
)

latex_data["Adjusted $p$"] = latex_data["holm_adjusted_p"].apply(
    lambda value: (
        "$<0.001$"
        if value < 0.001
        else f"{value:.3f}"
    )
)

latex_table = latex_data[
    [
        "question_type",
        "Stack Overflow, n (\\%)",
        "GitHub, n (\\%)",
        "OR (95\\% CI)",
        "Adjusted $p$",
    ]
].rename(
    columns={
        "question_type": "Question type",
    }
).to_latex(
    index=False,
    escape=False,
    caption=(
        "Cross-platform comparison of question types. "
        "Odds ratios greater than one indicate stronger "
        "association with Stack Overflow."
    ),
    label="tab:cross_platform_intent",
)

with open(
    OUTPUT_DIR / "cross_platform_table.tex",
    "w",
    encoding="utf-8",
) as latex_file:
    latex_file.write(latex_table)

print(
    "\nAnalysis completed. Results were saved in:",
    OUTPUT_DIR.resolve(),
)



Observed counts
question_type    how   why  what  other
platform                               
Stack Overflow   253    71   132     39
GitHub          2349  3998  1774    995

Platform percentages
question_type     how    why   what  other
platform                                  
Stack Overflow  51.11  14.34  26.67   7.88
GitHub          25.77  43.86  19.46  10.91

Overall cross-platform test
Chi-square: 224.2994
Degrees of freedom: 3
p-value: 2.3620295e-48
Cramér's V: 0.1528

Adjusted standardized residuals
question_type      how     why   what  other
platform                                    
Stack Overflow  12.359 -12.943  3.916 -2.123
GitHub         -12.359  12.943 -3.916  2.123

Question-type effect sizes
question_type  so_n  so_percent  github_n  github_percent  difference_percentage_points  odds_ratio_so_vs_github  ci_95_lower  ci_95_upper  p_value  holm_adjusted_p  significant_after_holm
          How   253     51.1111      2349         25.7679                       25.34